# E10a - Q-Learning and Deep Q-Networks (Exercises)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Hands-on exercises: from tabular Q-Learning to Deep Q-Networks (DQN)**

---

## Learning Objectives

- Implement tabular Q-Learning from scratch
- Understand the Bellman equation and temporal difference learning
- Build a Deep Q-Network (DQN) with experience replay and target networks
- Apply epsilon-greedy exploration strategies
- Train agents on classic control environments
- Compare Q-Learning vs DQN performance

**Duration:** 4-5 hours  
**Prerequisites:** E10 (Deep Reinforcement Learning concepts), PyTorch basics

**Relevant UoA Courses:** COMPSCI 761

---

## Part 1: Reinforcement Learning Foundations

### The RL Framework

In reinforcement learning, an **agent** interacts with an **environment**:

1. Agent observes **state** $s_t$
2. Agent takes **action** $a_t$
3. Environment returns **reward** $r_t$ and next state $s_{t+1}$
4. Goal: maximize cumulative discounted reward $G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$

### The Q-Function

The **action-value function** (Q-function) estimates expected return:

$$Q(s, a) = \mathbb{E}[G_t | s_t = s, a_t = a]$$

### Bellman Equation

$$Q(s, a) = r + \gamma \cdot \max_{a'} Q(s', a')$$

This recursive relationship is the foundation of Q-Learning.

---

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque, namedtuple
import random
import gymnasium as gym

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

print(f'PyTorch version: {torch.__version__}')
print(f'Gymnasium version: {gym.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

---

## Part 2: Tabular Q-Learning

### Exercise 1: Q-Learning on FrozenLake

We'll start with a simple grid world where the state and action spaces are small enough
to store Q-values in a table.

**FrozenLake-v1**: Navigate a 4x4 grid from Start (S) to Goal (G), avoiding Holes (H).

```
SFFF
FHFH
FFFH
HFFG
```

Actions: 0=Left, 1=Down, 2=Right, 3=Up

In [ ]:
# Create the FrozenLake environment (deterministic for easier learning)
env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False)

n_states = env.observation_space.n
n_actions = env.action_space.n

print(f'Number of states: {n_states}')
print(f'Number of actions: {n_actions}')
print(f'State space: {env.observation_space}')
print(f'Action space: {env.action_space}')

### Exercise 1a: Implement the Q-Learning Algorithm

The Q-Learning update rule:

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[ r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) \right]$$

Where:
- $\alpha$ = learning rate
- $\gamma$ = discount factor
- $r_t + \gamma \max_{a'} Q(s_{t+1}, a')$ = TD target
- $r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)$ = TD error

In [ ]:
class QLearningAgent:
    """Tabular Q-Learning agent."""
    
    def __init__(self, n_states, n_actions, learning_rate=0.1, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Initialize Q-table with zeros
        self.q_table = np.zeros((n_states, n_actions))
    
    def choose_action(self, state):
        """Epsilon-greedy action selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)  # Explore
        else:
            return np.argmax(self.q_table[state])  # Exploit
    
    def update(self, state, action, reward, next_state, done):
        """Q-Learning update rule."""
        # TD target
        if done:
            td_target = reward
        else:
            td_target = reward + self.gamma * np.max(self.q_table[next_state])
        
        # TD error
        td_error = td_target - self.q_table[state, action]
        
        # Update Q-value
        self.q_table[state, action] += self.lr * td_error
        
        return td_error
    
    def decay_epsilon(self):
        """Decay exploration rate."""
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

### Exercise 1b: Train the Q-Learning Agent

In [ ]:
def train_q_learning(env, agent, n_episodes=2000, max_steps=100):
    """Train a Q-Learning agent."""
    rewards_per_episode = []
    td_errors = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        episode_errors = []
        
        for step in range(max_steps):
            action = agent.choose_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            error = agent.update(state, action, reward, next_state, done)
            episode_errors.append(abs(error))
            
            total_reward += reward
            state = next_state
            
            if done:
                break
        
        agent.decay_epsilon()
        rewards_per_episode.append(total_reward)
        td_errors.append(np.mean(episode_errors) if episode_errors else 0)
        
        if (episode + 1) % 500 == 0:
            avg_reward = np.mean(rewards_per_episode[-100:])
            print(f'Episode {episode+1}/{n_episodes} | '
                  f'Avg Reward (last 100): {avg_reward:.3f} | '
                  f'Epsilon: {agent.epsilon:.4f}')
    
    return rewards_per_episode, td_errors

# Train the agent
agent = QLearningAgent(n_states, n_actions, learning_rate=0.8, gamma=0.95)
rewards, errors = train_q_learning(env, agent, n_episodes=2000)

### Exercise 1c: Visualize Training Progress and Q-Table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot rewards
window = 50
rolling_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
axes[0].plot(rolling_avg)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Average Reward')
axes[0].set_title('Training Rewards (Rolling Average)')
axes[0].axhline(y=1.0, color='g', linestyle='--', alpha=0.5, label='Optimal')
axes[0].legend()

# Plot TD errors
rolling_errors = np.convolve(errors, np.ones(window)/window, mode='valid')
axes[1].plot(rolling_errors)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Mean |TD Error|')
axes[1].set_title('TD Error Convergence')

# Visualize Q-table as heatmap
import seaborn as sns
action_labels = ['Left', 'Down', 'Right', 'Up']
sns.heatmap(agent.q_table, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=action_labels, ax=axes[2])
axes[2].set_xlabel('Action')
axes[2].set_ylabel('State')
axes[2].set_title('Learned Q-Table')

plt.tight_layout()
plt.show()

### Exercise 1d: Evaluate the Learned Policy

Let's see how the trained agent performs by running it greedily (no exploration).

In [ ]:
def evaluate_agent(env, agent, n_episodes=100):
    """Evaluate agent with greedy policy (no exploration)."""
    successes = 0
    for _ in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = np.argmax(agent.q_table[state])  # Greedy
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            if reward == 1.0:
                successes += 1
    return successes / n_episodes

success_rate = evaluate_agent(env, agent)
print(f'Success rate: {success_rate*100:.1f}%')

# Visualize the learned policy on the grid
policy = np.argmax(agent.q_table, axis=1)
arrows = ['\u2190', '\u2193', '\u2192', '\u2191']  # Left, Down, Right, Up
grid_map = ['SFFF', 'FHFH', 'FFFH', 'HFFG']

print('\nLearned Policy:')
print('-' * 20)
for i in range(4):
    row = ''
    for j in range(4):
        state_idx = i * 4 + j
        cell = grid_map[i][j]
        if cell in ['H', 'G']:
            row += f' {cell}  '
        else:
            row += f' {arrows[policy[state_idx]]}  '
    print(row)
print('-' * 20)

---

## Part 3: Q-Learning with Stochastic Transitions

### Exercise 2: Slippery FrozenLake

Now let's try the stochastic version where the ice is slippery.
The agent only moves in the intended direction 1/3 of the time!

**Challenge:** How does stochasticity affect learning? What hyperparameters need tuning?

In [ ]:
# Slippery version - much harder!
env_slippery = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True)

# Train with adjusted hyperparameters for stochastic environment
agent_slippery = QLearningAgent(
    n_states, n_actions,
    learning_rate=0.1,      # Lower LR for noisy updates
    gamma=0.99,             # Higher discount for long-term planning
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.999     # Slower decay - need more exploration
)

rewards_slippery, errors_slippery = train_q_learning(
    env_slippery, agent_slippery, n_episodes=10000
)

# Evaluate
success_rate_slippery = evaluate_agent(env_slippery, agent_slippery, n_episodes=1000)
print(f'\nSlippery FrozenLake success rate: {success_rate_slippery*100:.1f}%')
print(f'(Note: ~74% is near-optimal for slippery FrozenLake)')

### Exercise 2b: Hyperparameter Sensitivity Analysis

**Your Task:** Experiment with different learning rates and discount factors.
Fill in the code below to run a grid search.

In [ ]:
# TODO: Experiment with different hyperparameters
# Try different combinations and observe the effect

learning_rates = [0.01, 0.1, 0.5, 0.9]
results = {}

for lr in learning_rates:
    env_test = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False)
    test_agent = QLearningAgent(n_states, n_actions, learning_rate=lr, gamma=0.95)
    test_rewards, _ = train_q_learning(env_test, test_agent, n_episodes=1000)
    results[lr] = np.mean(test_rewards[-100:])
    env_test.close()

# Plot results
plt.figure(figsize=(8, 4))
plt.bar([str(lr) for lr in results.keys()], results.values(), color='steelblue')
plt.xlabel('Learning Rate')
plt.ylabel('Average Reward (last 100 episodes)')
plt.title('Effect of Learning Rate on Q-Learning Performance')
plt.ylim(0, 1.1)
plt.show()

print('\nResults:')
for lr, reward in results.items():
    print(f'  LR={lr}: avg reward = {reward:.3f}')

---

## Part 4: Deep Q-Networks (DQN)

### Why DQN?

Tabular Q-Learning fails when:
- State space is continuous (e.g., CartPole position/velocity)
- State space is too large to enumerate

**Solution:** Approximate $Q(s, a)$ with a neural network $Q(s, a; \theta)$

### Key DQN Innovations (Mnih et al., 2015)

1. **Experience Replay**: Store transitions in a buffer, sample random mini-batches
   - Breaks correlation between consecutive samples
   - Improves data efficiency

2. **Target Network**: Separate network for computing TD targets
   - Updated periodically (every C steps)
   - Stabilizes training by reducing moving target problem

### Exercise 3: Implement DQN Components

---

### Exercise 3a: Experience Replay Buffer

In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

class ReplayBuffer:
    """Fixed-size buffer to store experience tuples."""
    
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        """Add a new experience to the buffer."""
        self.buffer.append(Transition(state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        """Randomly sample a batch of experiences."""
        transitions = random.sample(self.buffer, batch_size)
        batch = Transition(*zip(*transitions))
        
        states = torch.FloatTensor(np.array(batch.state))
        actions = torch.LongTensor(batch.action)
        rewards = torch.FloatTensor(batch.reward)
        next_states = torch.FloatTensor(np.array(batch.next_state))
        dones = torch.FloatTensor(batch.done)
        
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.buffer)

# Test the replay buffer
buffer = ReplayBuffer(capacity=1000)
buffer.push([1, 2, 3, 4], 0, 1.0, [1, 2, 3, 5], False)
buffer.push([1, 2, 3, 5], 1, 0.0, [1, 2, 4, 5], True)
print(f'Buffer size: {len(buffer)}')
print(f'Sample transition: {buffer.buffer[0]}')

### Exercise 3b: The DQN Network Architecture

In [ ]:
class DQN(nn.Module):
    """Deep Q-Network: maps states to Q-values for each action."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
    
    def forward(self, x):
        return self.network(x)

# Test the network
state_dim = 4  # CartPole has 4 state variables
action_dim = 2  # CartPole has 2 actions (left/right)

policy_net = DQN(state_dim, action_dim)
target_net = DQN(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict())  # Copy weights

# Test forward pass
dummy_state = torch.randn(1, state_dim)
q_values = policy_net(dummy_state)
print(f'Input state shape: {dummy_state.shape}')
print(f'Output Q-values: {q_values.detach().numpy()}')
print(f'Selected action: {q_values.argmax().item()}')
print(f'\nNetwork architecture:\n{policy_net}')

### Exercise 3c: The Complete DQN Agent

In [ ]:
class DQNAgent:
    """Deep Q-Network agent with experience replay and target network."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=128,
                 lr=1e-3, gamma=0.99, epsilon_start=1.0,
                 epsilon_end=0.01, epsilon_decay=0.995,
                 buffer_size=10000, batch_size=64,
                 target_update_freq=10):
        
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.steps_done = 0
        
        # Networks
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.policy_net = DQN(state_dim, action_dim, hidden_dim).to(self.device)
        self.target_net = DQN(state_dim, action_dim, hidden_dim).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()  # Target net is not trained directly
        
        # Optimizer
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        
        # Replay buffer
        self.memory = ReplayBuffer(capacity=buffer_size)
    
    def choose_action(self, state):
        """Epsilon-greedy action selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.action_dim)
        else:
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                q_values = self.policy_net(state_tensor)
                return q_values.argmax().item()
    
    def store_transition(self, state, action, reward, next_state, done):
        """Store transition in replay buffer."""
        self.memory.push(state, action, reward, next_state, done)
    
    def learn(self):
        """Sample from replay buffer and perform gradient descent."""
        if len(self.memory) < self.batch_size:
            return None
        
        # Sample batch
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)
        
        # Current Q-values: Q(s, a) for the actions taken
        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        # Target Q-values: r + gamma * max_a' Q_target(s', a')
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1)[0]
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        # Loss: MSE between current and target Q-values
        loss = F.mse_loss(current_q, target_q)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        
        self.steps_done += 1
        
        # Update target network periodically
        if self.steps_done % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        return loss.item()
    
    def decay_epsilon(self):
        """Decay exploration rate."""
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

print('DQNAgent class defined successfully!')

---

## Part 5: Training DQN on CartPole-v1

### Exercise 4: Train DQN

**CartPole-v1**: Balance a pole on a cart by moving left/right.
- State: [cart_position, cart_velocity, pole_angle, pole_angular_velocity]
- Actions: 0 (push left), 1 (push right)
- Reward: +1 for each timestep the pole stays upright
- Solved: Average reward >= 475 over 100 episodes

In [ ]:
def train_dqn(env, agent, n_episodes=500, max_steps=500, print_every=50):
    """Train a DQN agent."""
    rewards_per_episode = []
    losses = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        episode_losses = []
        
        for step in range(max_steps):
            action = agent.choose_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store transition
            agent.store_transition(state, action, reward, next_state, float(done))
            
            # Learn from replay buffer
            loss = agent.learn()
            if loss is not None:
                episode_losses.append(loss)
            
            total_reward += reward
            state = next_state
            
            if done:
                break
        
        agent.decay_epsilon()
        rewards_per_episode.append(total_reward)
        if episode_losses:
            losses.append(np.mean(episode_losses))
        
        if (episode + 1) % print_every == 0:
            avg_reward = np.mean(rewards_per_episode[-100:])
            print(f'Episode {episode+1}/{n_episodes} | '
                  f'Avg Reward: {avg_reward:.1f} | '
                  f'Epsilon: {agent.epsilon:.4f} | '
                  f'Buffer: {len(agent.memory)}')
            
            if avg_reward >= 475:
                print(f'\n*** Solved in {episode+1} episodes! ***')
                break
    
    return rewards_per_episode, losses

# Create CartPole environment
cartpole_env = gym.make('CartPole-v1')

# Create DQN agent
dqn_agent = DQNAgent(
    state_dim=4,
    action_dim=2,
    hidden_dim=128,
    lr=1e-3,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.995,
    buffer_size=10000,
    batch_size=64,
    target_update_freq=10
)

# Train!
dqn_rewards, dqn_losses = train_dqn(cartpole_env, dqn_agent, n_episodes=500)

### Exercise 4b: Visualize DQN Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot rewards
axes[0].plot(dqn_rewards, alpha=0.3, color='blue', label='Episode Reward')
window = 20
if len(dqn_rewards) >= window:
    rolling = np.convolve(dqn_rewards, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(dqn_rewards)), rolling, color='red', 
                linewidth=2, label=f'Rolling Avg ({window} ep)')
axes[0].axhline(y=475, color='green', linestyle='--', alpha=0.7, label='Solved (475)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('DQN Training on CartPole-v1')
axes[0].legend()

# Plot loss
if dqn_losses:
    axes[1].plot(dqn_losses, alpha=0.5, color='orange')
    if len(dqn_losses) >= window:
        rolling_loss = np.convolve(dqn_losses, np.ones(window)/window, mode='valid')
        axes[1].plot(range(window-1, len(dqn_losses)), rolling_loss, 
                    color='red', linewidth=2)
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('DQN Training Loss')

plt.tight_layout()
plt.show()

---

## Part 6: DQN Improvements

### Exercise 5: Double DQN

Standard DQN overestimates Q-values because it uses the same network to both
**select** and **evaluate** the best action.

**Double DQN** (van Hasselt et al., 2016) decouples selection from evaluation:

- Standard DQN target: $r + \gamma \cdot Q_{target}(s', \arg\max_{a'} Q_{target}(s', a'))$
- Double DQN target: $r + \gamma \cdot Q_{target}(s', \arg\max_{a'} Q_{policy}(s', a'))$

The policy network selects the action, but the target network evaluates it.

In [ ]:
class DoubleDQNAgent(DQNAgent):
    """Double DQN - reduces overestimation of Q-values."""
    
    def learn(self):
        """Double DQN learning step."""
        if len(self.memory) < self.batch_size:
            return None
        
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)
        
        # Current Q-values
        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        # Double DQN: use policy net to SELECT action, target net to EVALUATE
        with torch.no_grad():
            # Policy net selects best action for next state
            best_actions = self.policy_net(next_states).argmax(1)
            # Target net evaluates Q-value of that action
            next_q = self.target_net(next_states).gather(1, best_actions.unsqueeze(1)).squeeze(1)
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        loss = F.mse_loss(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        
        self.steps_done += 1
        if self.steps_done % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        return loss.item()

# Train Double DQN
cartpole_env2 = gym.make('CartPole-v1')
double_dqn_agent = DoubleDQNAgent(
    state_dim=4, action_dim=2, hidden_dim=128,
    lr=1e-3, gamma=0.99, epsilon_decay=0.995,
    buffer_size=10000, batch_size=64, target_update_freq=10
)

print('Training Double DQN...')
ddqn_rewards, ddqn_losses = train_dqn(cartpole_env2, double_dqn_agent, n_episodes=500)

### Exercise 5b: Compare DQN vs Double DQN

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

window = 20

# DQN
if len(dqn_rewards) >= window:
    rolling_dqn = np.convolve(dqn_rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(dqn_rewards)), rolling_dqn, 
            label='DQN', linewidth=2, color='blue')

# Double DQN
if len(ddqn_rewards) >= window:
    rolling_ddqn = np.convolve(ddqn_rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(ddqn_rewards)), rolling_ddqn, 
            label='Double DQN', linewidth=2, color='green')

ax.axhline(y=475, color='red', linestyle='--', alpha=0.7, label='Solved (475)')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (Rolling Average)')
ax.set_title('DQN vs Double DQN on CartPole-v1')
ax.legend()
plt.tight_layout()
plt.show()

print(f'DQN final avg reward: {np.mean(dqn_rewards[-50:]):.1f}')
print(f'Double DQN final avg reward: {np.mean(ddqn_rewards[-50:]):.1f}')

---

## Part 7: Dueling DQN Architecture

### Exercise 6: Implement Dueling DQN

**Dueling DQN** (Wang et al., 2016) separates the Q-function into:

$$Q(s, a) = V(s) + A(s, a) - \frac{1}{|A|} \sum_{a'} A(s, a')$$

Where:
- $V(s)$ = state value (how good is this state?)
- $A(s, a)$ = advantage (how much better is action $a$ than average?)

This helps the network learn which states are valuable without needing to
learn the effect of each action in every state.

In [ ]:
class DuelingDQN(nn.Module):
    """Dueling DQN: separate value and advantage streams."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(DuelingDQN, self).__init__()
        
        # Shared feature layer
        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Value stream: V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Advantage stream: A(s, a)
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, action_dim)
        )
    
    def forward(self, x):
        features = self.feature(x)
        value = self.value_stream(features)
        advantage = self.advantage_stream(features)
        # Q = V + (A - mean(A))
        q_values = value + advantage - advantage.mean(dim=1, keepdim=True)
        return q_values

# Test Dueling DQN
dueling_net = DuelingDQN(state_dim=4, action_dim=2)
dummy_state = torch.randn(1, 4)
q_vals = dueling_net(dummy_state)
print(f'Dueling DQN output: {q_vals.detach().numpy()}')
print(f'\nArchitecture:\n{dueling_net}')

---

## Part 8: Challenge Exercises

### Exercise 7: DQN on LunarLander

**Your Task:** Apply DQN to a harder environment - LunarLander-v3.

- State: 8-dimensional (position, velocity, angle, leg contact)
- Actions: 4 (do nothing, fire left, fire main, fire right)
- Solved: Average reward >= 200 over 100 episodes

Hints:
- You may need more episodes (1000+)
- Try a larger network (256 hidden units)
- Slower epsilon decay helps exploration

In [ ]:
# TODO: Implement DQN for LunarLander-v3
# This is a challenge exercise - modify the hyperparameters and train!

lunar_env = gym.make('LunarLander-v3')

lunar_agent = DQNAgent(
    state_dim=8,
    action_dim=4,
    hidden_dim=256,         # Larger network
    lr=5e-4,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.998,    # Slower decay
    buffer_size=50000,      # Larger buffer
    batch_size=64,
    target_update_freq=20
)

print('Training DQN on LunarLander-v3...')
print('(This may take a few minutes)')
lunar_rewards, lunar_losses = train_dqn(
    lunar_env, lunar_agent, n_episodes=800, max_steps=1000, print_every=100
)

# Plot results
plt.figure(figsize=(10, 4))
window = 50
if len(lunar_rewards) >= window:
    rolling = np.convolve(lunar_rewards, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(lunar_rewards)), rolling, linewidth=2)
plt.axhline(y=200, color='green', linestyle='--', label='Solved (200)')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('DQN on LunarLander-v3')
plt.legend()
plt.show()

print(f'Final avg reward (last 100): {np.mean(lunar_rewards[-100:]):.1f}')

### Exercise 8: Ablation Study

**Your Task:** Investigate the importance of each DQN component.

Run experiments removing one component at a time:
1. No experience replay (learn from most recent transition only)
2. No target network (use policy net for targets)
3. No epsilon decay (fixed epsilon=0.1)

Which component matters most?

In [ ]:
# Ablation: No target network (use policy net for both)
class DQNNoTarget(DQNAgent):
    """DQN without target network - demonstrates instability."""
    
    def learn(self):
        if len(self.memory) < self.batch_size:
            return None
        
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)
        
        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        # Use POLICY net (not target) for next Q-values - less stable!
        with torch.no_grad():
            next_q = self.policy_net(next_states).max(1)[0]
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        loss = F.mse_loss(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        self.steps_done += 1
        return loss.item()

# Train without target network
cartpole_env3 = gym.make('CartPole-v1')
no_target_agent = DQNNoTarget(
    state_dim=4, action_dim=2, hidden_dim=128,
    lr=1e-3, gamma=0.99, epsilon_decay=0.995,
    buffer_size=10000, batch_size=64, target_update_freq=10
)

print('Training DQN WITHOUT target network...')
no_target_rewards, _ = train_dqn(cartpole_env3, no_target_agent, n_episodes=300)

# Compare
plt.figure(figsize=(10, 4))
window = 20
if len(dqn_rewards) >= window:
    r1 = np.convolve(dqn_rewards[:300], np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(r1)+window-1), r1, label='DQN (with target net)', linewidth=2)
if len(no_target_rewards) >= window:
    r2 = np.convolve(no_target_rewards, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(r2)+window-1), r2, label='DQN (no target net)', linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Ablation: Effect of Target Network')
plt.legend()
plt.show()

---

## Summary and Key Takeaways

### What We Covered

| Topic | Key Idea |
|-------|----------|
| Q-Learning | Tabular TD learning with Bellman updates |
| Epsilon-Greedy | Balance exploration vs exploitation |
| DQN | Neural network approximates Q-function |
| Experience Replay | Break correlation, improve sample efficiency |
| Target Network | Stabilize training with frozen targets |
| Double DQN | Reduce Q-value overestimation |
| Dueling DQN | Separate value and advantage streams |

### Key Equations

1. **Q-Learning Update:** $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$
2. **DQN Loss:** $L = \mathbb{E}[(r + \gamma \max_{a'} Q_{target}(s',a') - Q(s,a))^2]$
3. **Double DQN Target:** $y = r + \gamma Q_{target}(s', \arg\max_{a'} Q_{policy}(s', a'))$

### Common Pitfalls

- ❌ Learning rate too high → unstable Q-values
- ❌ Buffer too small → correlated samples
- ❌ Epsilon decays too fast → insufficient exploration
- ❌ No target network → moving target problem
- ❌ No gradient clipping → exploding gradients

### Further Reading

- Mnih et al. (2015) - "Human-level control through deep reinforcement learning" (Nature)
- van Hasselt et al. (2016) - "Deep Reinforcement Learning with Double Q-learning"
- Wang et al. (2016) - "Dueling Network Architectures for Deep RL"
- Schaul et al. (2016) - "Prioritized Experience Replay"

---

---

## Learning Progress Tracker

### Completion Status
- [ ] Part 1-2: Tabular Q-Learning on FrozenLake
- [ ] Part 3: Stochastic environment and hyperparameter tuning
- [ ] Part 4-5: DQN implementation and CartPole training
- [ ] Part 6: Double DQN
- [ ] Part 7: Dueling DQN architecture
- [ ] Part 8: Challenge exercises (LunarLander, Ablation)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____

### Understanding Level
Rate your understanding (1-5): _____ / 5

### Notes & Reflections
```
Write your notes here:
- What was the hardest part?
- How does epsilon decay affect learning?
- When would you use tabular Q-Learning vs DQN?
- What other improvements could you try?


```

---